# Block web-search domains and redact matching response URLs

This lab shows how to add organization domains to a Responses API request while preserving the caller's existing blocklist. APIM only applies the merge when a `web_search` tool is present. A separate regex Azure Function replaces complete blocked URLs with `[BLOCKED LINK]` only after the response actually contains `web_search_call`, for JSON and streaming responses. For example, `https://youtube.com/watch?v=1` becomes `[BLOCKED LINK]`; surrounding text, link labels, and markup stay intact.

For developers familiar with APIM and Foundry. You need an existing APIM service, a Foundry backend or resource endpoint, an existing model deployment that supports `web_search`, Azure CLI authentication, and the repository's Python environment (`uv sync` at the repo root).

1. Load existing resource settings from `.env`.
2. Inspect the policy and prepare routing to the existing backend.
3. Deploy the lab API.
4. Send requests with and without web search and inspect the backend request in APIM tracing.
5. Compare JSON and streaming metrics using the same `x-response-metrics` JSON map, `x-response-metrics-status`, and `x-response-request-id` headers.

See [README.md](README.md) for configuration and policy behavior, and [Microsoft's domain-filtering documentation](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/web-search#domain-filtering) for the request format.

## 1. Load configuration

Run with this lab directory as the working directory. The loader reads the root `.env`, then this lab's `.env`; process variables win. To select another existing file, set `env_file` below or use `LAB_ENV_FILE`.

`BACKEND_ID` / `APIM_BACKEND_ID` reuses an existing APIM Foundry backend and takes precedence over `AZURE_OPENAI_ENDPOINT`. A resource endpoint creates only a backend on existing APIM. Neither option provisions Foundry or a model deployment. Avoid printing configuration dictionaries because they can contain keys.

Every prompt below includes the shared `system_prompt` as a system message. It instructs the model to avoid the `site:` operator in all `web_search` queries.

In [ ]:
import importlib
from src import lab

importlib.reload(lab)
from src.lab import load_config, prepare_deployment, send_response, show_response_metrics

system_prompt = (
    "When using the web_search tool, never use the site: operator in any search query "
    "(including site:example.com or -site:example.com). Use natural-language search "
    "terms instead and respect the configured domain filters."
)

# Example: env_file = "../secure-responses-api/.env"
env_file = None
config = load_config(env_file)
model = config.get("AZURE_OPENAI_DEPLOYMENT") or config.get("AZURE_OPENAI_DEPLOYMENT_NAME")
if not model:
    raise ValueError("Set AZURE_OPENAI_DEPLOYMENT to an existing deployment supporting web_search.")
print("Configuration loaded; model deployment:", model)


## 2. Inspect the policy and existing backend

The first `<choose>` in [policy.xml](policy.xml) checks `tools` using a preserved copy of the request body. Its branch validates the web-search filters and appends missing domains, preserving existing entries and unrelated fields. The organization blocklist comes from [blocked-domains.json](blocked-domains.json); the same file configures the regex Function. Redaction uses only blocked domains and treats every other hostname as allowed.

`web_search_preview` does not support this filtering contract and is left unchanged. Use `web_search` here.

The next cell performs read-only Azure lookups. It fails if a configured resource has been deleted. For backend pools or custom URLs, supply `BACKEND_RESPONSES_PATH` as described in the README.

In [ ]:
prepared = prepare_deployment(config)
print("APIM service:", prepared["parameters"]["apimServiceName"])
print("Existing backend:", prepared["parameters"]["backendId"] or "Create lab backend for configured endpoint")
print("Backend Responses path:", prepared["parameters"]["backendResponsesPath"])
print("Lab Responses URL:", prepared["responses_url"])

## 3. Deploy the dedicated lab API

The next cell deploys the HTTP-only proxy and the separate `/api/redact` function in the same Function App, grants its managed identity Foundry access, publishes its code, and configures APIM after both routes pass readiness checks. The regex probe uses a synthetic response and makes no model call. It also grants APIM's system-assigned identity **Cognitive Services OpenAI User** for direct JSON requests. An endpoint configuration with `AZURE_OPENAI_API_KEY` uses key authentication instead.

Choose an unused `APIM_API_NAME` / `APIM_API_PATH` for the first run. The CLI account needs deployment and role-assignment permissions. Set `AZURE_OPENAI_RESOURCE_ID` for an account in another subscription or with a custom hostname; see the README for backend-pool settings. Role grants may take a few minutes to propagate.

The Function runs on one Linux B1 instance with filesystem host keys and no Azure Storage account. This is experimental; Microsoft documents host storage as required. The plan incurs charges while retained. See [proxy configuration](README.md#proxy-deployment-and-configuration) for limits.

The deployment cell reloads `src.lab` before calling `lab.deploy`, so a running kernel picks up helper and template changes together.


In [ ]:
import importlib
from src import lab

# Read the current helper before deploying; earlier imported functions can be stale.
importlib.reload(lab)
deployment = lab.deploy(config, prepared)
print("Deployment state:", deployment["properties"]["provisioningState"])
print("Responses URL:", prepared["responses_url"])

## 4. Control request: no web-search tool

This makes a model request through APIM with no `tools` property. Blocklist validation and `set-body` are skipped. The example prints the gateway metric headers used by every request: `x-response-metrics` contains `web_search_count` from `tool_usage.web_search.num_requests` (zero only when explicitly reported by the service) and `total_tokens`. Missing values are `null`; the status is `reported`, `partial`, or `unavailable`. In the portal Test tab, enable tracing and verify that the backend request has no added `tools` or `filters` properties.

In [ ]:
def show_text(response):
    for item in response.get("output", []):
        for content in item.get("content", []):
            if content.get("type") == "output_text":
                print(content["text"])

without_web_search = {
    "model": model,
    "input": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "Reply with the word hello."},
    ],
    "store": False,
}
control_response, control_headers = send_response(
    config, prepared, without_web_search, return_headers=True,
)
show_response_metrics(control_headers)
show_text(control_response)

## 5. Web search without an existing blocklist

Only the tool declaration is sent by the client. APIM adds `filters.blocked_domains` with the 11 organization domains. `tool_choice` requests a web-search call, and `include` asks Foundry to return consulted sources. Inspect `web_search_count` in the printed `x-response-metrics` JSON map; it reads `tool_usage.web_search.num_requests`, just like the streaming proxy. Missing usage is `null`; output queries are not counted.

For JSON, the proxy reads the complete search response, checks for `web_search_call`, then invokes `/api/redact`. Complete answer links, citation URLs, and search-source URLs are replaced with `[BLOCKED LINK]`. An offered but unused tool skips the regex Function and returns the original response.

In [ ]:
with_web_search = {
    "model": model,
    "input": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "Find youtube videos for AI advancements"},
    ],
    "tools": [{"type": "web_search"}],
    "tool_choice": {"type": "web_search"},
    "include": ["web_search_call.action.sources"],
    "store": False,
}
search_response, search_headers = send_response(
    config, prepared, with_web_search, return_headers=True,
)
show_response_metrics(search_headers)
show_text(search_response)

## 6. Preserve the caller's blocklist and other filters

The client blocks `example.com` and `youtube.com`. The gateway must preserve `example.com`, keep a single `youtube.com`, and append the other 10 domains. It must also retain the caller's `allowed_domains` and `search_context_size`.

Run this request, then repeat it in APIM's portal Test tab with tracing enabled. Check the **Backend request body**, rather than relying on the response to echo tool settings. The configured domains are listed in the README. The response prints the same metric headers as the control and streaming examples.

In [ ]:
from copy import deepcopy

with_existing_blocklist = deepcopy(with_web_search)
with_existing_blocklist["tools"][0].update({
    "search_context_size": "low",
    "filters": {
        "blocked_domains": ["example.com", "youtube.com"],
    },
})
merged_response, merged_headers = send_response(
    config, prepared, with_existing_blocklist, return_headers=True,
)
show_response_metrics(merged_headers)
show_text(merged_response)

# This is the client-side request. APIM adds domains only after receiving it.
assert with_existing_blocklist["tools"][0]["filters"]["blocked_domains"] == ["example.com", "youtube.com"]

## 7. Inspect returned search sources

Sources provide a useful observation of the search result. They do not prove that APIM transformed every request correctly; use APIM tracing for that.

In [ ]:
for item in merged_response.get("output", []):
    if item.get("type") == "web_search_call":
        for source in item.get("action", {}).get("sources", []):
            print(source.get("url", ""))

## 8. Stream through the Function and inspect completion metrics

Run sections 1–3 first. The regex Function is called only after actual search invocation.

After search starts, consecutive `response.output_text.delta` events for the same
message and content part are filtered in non-overlapping pairs. The relay holds
one delta until its partner arrives, joins their text for regex matching, then
returns both original SSE events in order. A URL spanning the pair becomes one
`[BLOCKED LINK]`; its placeholder stays in the first event and the second retains
any following text. IDs, sequence numbers, content indexes, and event counts are
preserved. An unmatched delta is filtered on its own before a non-text event,
a different content part, or an upstream comment. Terminal events flush it before
completion. Unused tools and events before search remain unchanged.

For example, `https://you` plus `tube.com/a` in one pair becomes `[BLOCKED LINK]`.
There is no carry-over between pairs: a URL spanning the second event of one pair
and the first of the next can still escape detection. URLs spanning three or more
deltas, or interrupted by other event types, can also be missed. Paths or queries
arriving in a later pair can remain, and a partial blocked hostname can be
replaced before a later pair adds an allowed suffix. Complete `done` and terminal
snapshots are filtered separately and can differ from concatenated deltas.
Citation offsets are corrected within complete text snapshots; standalone
annotation offsets still refer to the original text.

The relay assembles incomplete transport frames to preserve JSON and UTF-8. Each
frame and each redaction request (including both events) is limited to 2 MB;
oversized pairs fail with an SSE error. It never collects the full response.
Pairing adds a one-delta wait plus the regex HTTP call. `x-response-redaction-window: 2`
identifies this mode; the existing `buffered: false` headers mean no full-response
buffering, while one text delta can be waiting for its partner.

Initial headers contain `x-response-metrics-status: deferred`, `x-response-request-id`, `x-response-buffered: false`, and `x-response-redaction: conditional` for search-capable requests. These headers precede the actual invocation. At stream completion, the Function sends a separate report containing `web_search_count` and `total_tokens`. APIM logs the same metric map and request ID with status `proxy-reported`, `partial`, or `unavailable`.

The search count comes from `response.tool_usage.web_search.num_requests` in the terminal event. Add extractors in [proxy/metrics.py](proxy/metrics.py). Missing values become null. The terminal event is retained as `streamed_event`; local metrics below are only a comparison. Reporting retries up to three times within 15 seconds after events have been forwarded, so reporting can delay HTTP EOF but not model events. Process crashes or sustained failures can lose reports.

The example prints live search events and text deltas with their arrival times. Set `use_web_search = False` to compare an ordinary stream. See [logging queries](README.md#log-the-headers-in-apim) to correlate and deduplicate reports. If a partial upgrade returns **HTTP 400: Missing gateway request ID**, reload `src.lab` and run `lab.deploy(config, prepared, reuse_streaming_proxy=True)` to update APIM using the existing Function.

In [ ]:
import json
from time import perf_counter
import requests
from src.lab import show_response_metrics

use_web_search = True
streaming_request = {
    "model": model,
    "input": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "Find youtube videos for AI advancements"},
    ],
    "stream": True,
    "store": False,
    "reasoning": {"effort": "high"},
}
if use_web_search:
    streaming_request.update({
        "tools": [{"type": "web_search"}],
        "tool_choice": {"type": "web_search"},
        "include": ["web_search_call.action.sources"],
    })

subscription_key = config.get("APIM_SUBSCRIPTION_KEY")
if not subscription_key:
    raise ValueError("Set APIM_SUBSCRIPTION_KEY to an all-APIs or lab-API subscription key.")

started = perf_counter()
first_event_at = None
first_text_at = None
streamed_response = None
streamed_event = None
with requests.post(
    prepared["responses_url"],
    headers={"api-key": subscription_key, "Accept": "text/event-stream"},
    json=streaming_request, stream=True, timeout=(10, 180),
) as http_response:
    if not http_response.ok:
        detail = http_response.text
        for secret in (subscription_key, config.get("AZURE_OPENAI_API_KEY")):
            if secret:
                detail = detail.replace(secret, "<redacted>")
        message = f"HTTP {http_response.status_code}: {detail[:2000]}"
        request_id = (http_response.headers.get("x-response-request-id")
                      or http_response.headers.get("x-web-search-request-id"))
        if request_id:
            message += f"\nGateway request ID: {request_id}"
        if http_response.status_code == 400 and "Missing gateway request ID" in detail:
            message += (
                "\nThe Function requires x-response-request-id; APIM may still use the old header. "
                "Reload src.lab, then run lab.deploy(config, prepared, reuse_streaming_proxy=True) "
                "to update APIM using the existing Function, and retry this cell."
            )
        raise RuntimeError(message)
    if "text/event-stream" not in http_response.headers.get("Content-Type", "").lower():
        raise RuntimeError("Expected an SSE response; check that the backend supports stream=True.")

    print(f"Response headers received after {perf_counter() - started:.2f}s")
    show_response_metrics(http_response.headers)
    print("Redaction:", http_response.headers.get("x-response-redaction", "not needed"))
    print("Full-response buffered:", http_response.headers.get("x-response-buffered", "false"))
    print("Redaction window:", http_response.headers.get("x-response-redaction-window", "0"))
    metrics_request_id = http_response.headers.get("x-response-request-id")
    print()

    # Foundry Responses events carry one JSON object per SSE data line.
    # A small read size avoids adding client-side buffering to live streams.
    for line in http_response.iter_lines(chunk_size=1):
        if not line.startswith(b"data:"):
            continue  # Ignore event names, blank lines, and keepalive comments.
        data = line[5:].strip()
        if data == b"[DONE]":
            break
        if not data:
            continue
        event = json.loads(data)
        event_type = event.get("type")
        if first_event_at is None:
            first_event_at = perf_counter() - started
            print(f"First SSE event ({event_type}) after {first_event_at:.2f}s", flush=True)
        if event_type in ("response.web_search_call.in_progress", "response.web_search_call.searching", "response.web_search_call.completed"):
            print(f"[{perf_counter() - started:.2f}s] {event_type}", flush=True)
        if event_type == "response.output_text.delta":
            if first_text_at is None:
                first_text_at = perf_counter() - started
            print(event.get("delta", ""), end="", flush=True)
        elif event_type in ("response.completed", "response.incomplete", "response.failed"):
            streamed_response = event["response"]
            streamed_event = event
        elif event_type == "error":
            raise RuntimeError(f"Streaming error: {event.get('message', event)}")

print()
if streamed_response is None:
    raise RuntimeError("The stream ended without a terminal response event; it may have been interrupted.")
print("Final status:", streamed_response.get("status"))
if first_text_at is not None:
    print(f"First text received after {first_text_at:.2f}s")
print(f"Total elapsed time: {perf_counter() - started:.2f}s")
if streamed_response.get("error"):
    print("Response error:", streamed_response["error"])
if streamed_response.get("incomplete_details"):
    print("Incomplete details:", streamed_response["incomplete_details"])

### Compare metrics from the completed response

This cell makes no network requests. It reloads the metric registry and accounting module, then uses the existing `streamed_event`. After editing the proxy modules or seeing an import error, rerun just this cell; the model request does not need to be repeated. The local comparison is serialized as JSON (including `null` for missing values), matching the `x-response-metrics` header shown by the other examples. It is not a confirmation that APIM received the proxy report.


In [ ]:
import importlib

# Reload in dependency order: the kernel may still have the pre-refactor modules.
importlib.invalidate_caches()
from proxy import metrics, accounting
importlib.reload(metrics)
importlib.reload(accounting)

local_metrics = accounting.ResponseMetrics().extract(streamed_event)
print("x-response-metrics (local comparison):", accounting.serialize_metrics(local_metrics))
print("Find the proxy metrics report in APIM logs using:", metrics_request_id)


## 9. Verify URL replacement locally

This check makes no Azure or model calls. It verifies the blocklist-only behavior and preservation of text outside the replaced URLs.

In [ ]:
from proxy.filtering import DomainRedactor

redactor = DomainRedactor(["youtube.com"])
text = "Keep  [video](https://youtube.com/watch?v=1) and https://example.com/docs.\n"
expected = "Keep  [video]([BLOCKED LINK]) and https://example.com/docs.\n"
assert redactor.text(text) == expected
print(redactor.text(text), end="")

from proxy.streaming import redact_events
first = {"type": "response.output_text.delta", "item_id": "msg_pair", "output_index": 1,
         "content_index": 0, "delta": "Before https://you"}
second = {**first, "delta": "tube.com/a after"}
events = redact_events({"events": [first, second],
                        "search_event": {"type": "response.web_search_call.searching"}},
                       DomainRedactor(["youtube.com"]))
assert [event["delta"] for event in events] == ["Before [BLOCKED LINK]", " after"]
print("Two text deltas filtered together:", "".join(event["delta"] for event in events))
